# Cross-encoder second scorer (run CE) on the run-4 candidate set

**train CE → score fold-B pairs + all run-4 test candidates → blend with LightGBM → output_ce/ → validate**

- Model: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (Apache-2.0, 117.7M parameters), fine-tuned
  as a binary pair classifier on train blocking pairs. Only the pretrained weights are downloaded; no external data.
- Settings: **Accelerator GPU T4 x2**, **Internet ON** (pip + model download).
- Inputs: the challenge data, the code dataset (new version with `src/cross_encoder.py`), and the **run-4 output of
  `run_kaggle.ipynb`** (its `cache/`: normalized data, features, model OOF, pre-filter keep sets, cached test
  probabilities; and `output/candidate_pairs.tsv`). The run-4 cache is read in place (read-only).
- Optional: the output of an earlier version of THIS notebook, to resume (its `cache_ce/` is copied back).
- Writes `/kaggle/working/output_ce/` (`matching_results.tsv`, `candidate_pairs.tsv` = run 4's candidates),
  `/kaggle/working/cache_ce/` (checkpoints: CE model, score shards) and `/kaggle/working/ce.log`.
- `output/` of run 4 is never touched: it stays the fallback package.

## Step 1: locate the inputs and set paths
Finds the challenge data, the code dataset (copied to `/kaggle/working` so it can be run), the run-4 `cache/`
(normalized data, features, matcher OOF, pre-filter keep set) and the cache of the earlier cross-encoder where needed,
prints every path and fails early with a clear message if an input is missing or too old.

In [ ]:
import glob, os, shutil, subprocess, sys, hashlib

found_data = sorted(glob.glob("/kaggle/input/**/train/train_source1.tsv", recursive=True))
assert found_data, "challenge data not found"
DATA_DIR = os.path.dirname(os.path.dirname(found_data[0]))
WORK = "/kaggle/working"
WORK_CODE = f"{WORK}/business_entity_resolution"
hits = sorted(glob.glob("/kaggle/input/**/business_entity_resolution/src/cross_encoder.py", recursive=True))
assert hits, "code dataset with src/cross_encoder.py not found (attach the new amlc26-code version)"
shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), WORK_CODE, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("__pycache__"))

# run-4 cache: the input whose cache holds the pre-filter test keep set and the cached test probabilities
caches = [os.path.dirname(os.path.dirname(p)) for p in
          sorted(glob.glob("/kaggle/input/**/cache/prefilter/test_keep.parquet", recursive=True))]
caches = [c for c in caches if os.path.exists(f"{c}/predict/test_probs_pf.parquet")]
assert caches, "run-4 output (cache/prefilter/test_keep.parquet + cache/predict/test_probs_pf.parquet) not attached"
RUN4_CACHE = caches[0]
RUN4_CAND = os.path.join(os.path.dirname(RUN4_CACHE), "output", "candidate_pairs.tsv")
CE_DIR, OUT_CE, LOG = f"{WORK}/cache_ce", f"{WORK}/output_ce", f"{WORK}/ce.log"

# resume: cache_ce of an earlier version of this notebook
prev = sorted(glob.glob("/kaggle/input/**/cache_ce/pairs_B.parquet", recursive=True))
if prev and not os.path.exists(f"{CE_DIR}/pairs_B.parquet"):
    shutil.copytree(os.path.dirname(prev[0]), CE_DIR, dirs_exist_ok=True, copy_function=shutil.copy2)
    print("resumed cache_ce from", os.path.dirname(prev[0]))

validators = sorted(glob.glob("/kaggle/input/**/utils/validate_submission.py", recursive=True))
os.environ.update(BER_DATA_DIR=DATA_DIR, BER_CACHE_DIR=RUN4_CACHE, PYTHONIOENCODING="utf-8",
                  TOKENIZERS_PARALLELISM="false")
print("data:      ", DATA_DIR)
print("run-4 cache:", RUN4_CACHE)
print("run-4 candidate file:", RUN4_CAND, os.path.exists(RUN4_CAND))
print("validator: ", validators[0] if validators else "NOT FOUND")

## Step 2: install the small extra packages
The GPU image already has torch / transformers / lightgbm / scikit-learn / numba; only the small pure-Python
packages pinned in `requirements.txt` are added.

In [ ]:
# Kaggle's GPU image already has torch / transformers / lightgbm / sklearn / numba; add only the small
# pure packages our modules import, without touching their dependencies
!pip install -q --no-deps rapidfuzz==3.14.6 anyascii==0.3.3 Metaphone==0.6 2>&1 | tail -2
!nvidia-smi --query-gpu=name,memory.total --format=csv && nproc && free -g | head -2
import torch, transformers; print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
                                  torch.cuda.device_count(), "| transformers", transformers.__version__)

## Step 3: reinstall polars cleanly and check it imports
The GPU image ships another polars build; it is removed and polars 1.44.2 is reinstalled with its runtime, then
imported in a subprocess. The notebook stops here if that fails (instead of failing an hour later).

In [ ]:
# The GPU image ships its own polars; a --no-deps install of polars==1.44.2 leaves it without its binary
# runtime ("Polars binary is missing!", NameError PyLazyFrame). Remove every polars package, then install
# 1.44.2 WITH its dependencies (it pulls the matching polars-runtime package), and verify in a fresh process.
!pip uninstall -y -q polars polars-lts-cpu polars-u64-idx polars-runtime-32 polars-runtime-64 polars-runtime-compat 2>&1 | tail -2
!pip install -q polars==1.44.2 2>&1 | tail -2
chk = subprocess.run([sys.executable, "-c", "import polars as pl; print('polars', pl.__version__); "
                      "print(pl.DataFrame({'a': [1, 2]}).lazy().select(pl.col('a').sum()).collect().item())"],
                     capture_output=True, text=True)
print(chk.stdout, chk.stderr[-2000:])
assert chk.returncode == 0 and "polars 1.44.2" in chk.stdout, "polars 1.44.2 is not importable: fix before running"


## Step 4: run the cross-encoder
Runs `cross_encoder.py`, which fine-tunes CE v1 (paraphrase-multilingual-MiniLM-L12-v2, Apache-2.0, 117.7M) on 1.2M fold-A pairs: stages pairs (fold A/B split by S1) -> texts -> train (1 epoch, fp16, both GPUs) -> score fold B -> score all 7.73M test candidates (resumable 500k-pair shards) -> blend (informational).
Stages are checkpointed in `cache_ce/` (pairs, texts, CE model, score shards of 500k pairs); rerunning this cell
skips finished stages. `N_TRAIN` / `MAX_LEN` trade accuracy for time (defaults 1.2M pairs, 96 tokens).

In [ ]:
N_TRAIN, MAX_LEN = 1_200_000, 96
cmd = (f"set -o pipefail; {sys.executable} -u {WORK_CODE}/src/cross_encoder.py --ce_dir {CE_DIR} --out {OUT_CE} "
       f"--n_train {N_TRAIN} --max_len {MAX_LEN} --train_bs 128 --score_bs 1024 --workers 4 2>&1 | tee -a {LOG}")
print(">>>", cmd, flush=True)
rc = subprocess.run(cmd, shell=True, executable="/bin/bash").returncode   # bash: /bin/sh has no pipefail
print("exit code:", rc)
if rc != 0:
    raise RuntimeError(f"cross_encoder.py failed with exit code {rc}: see the output above / {LOG}")

## Step 5: check the outputs
The candidate file written next to the blend must be byte-identical to run 4's (same pairs, same order); the
submission files are validated with `utils/validate_submission.py` when it is attached.

In [ ]:
# candidate file must be identical to run 4's (same pairs, same order and formatting)
def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""):
            h.update(chunk)
    return h.hexdigest()
if os.path.exists(RUN4_CAND):
    same = md5(RUN4_CAND) == md5(f"{OUT_CE}/candidate_pairs.tsv")
    print("output_ce/candidate_pairs.tsv identical to run 4's:", same)
    if not same:
        shutil.copy2(RUN4_CAND, f"{OUT_CE}/candidate_pairs.tsv")
        print("-> replaced by run 4's file (the blend only uses those pairs)")
if validators:
    out = subprocess.run([sys.executable, validators[0], "--matching", f"{OUT_CE}/matching_results.tsv",
                          "--candidate", f"{OUT_CE}/candidate_pairs.tsv", "--test-dir", f"{DATA_DIR}/test"],
                         capture_output=True, text=True)
    print(out.stdout[-1500:], out.stderr[-500:])
    open(LOG, "a").write(out.stdout)

## Step 6: summary
Timings of each stage, the fold-B table (LightGBM, cross-encoder, blends) and matches per S1 per country.

In [ ]:
# summary: timings, fold-B table, chosen blend, matches per S1 per country
log = open(LOG, encoding="utf-8").read()
keys = ("<<<", "=== Fold B", "LightGBM alone", "chosen:", "output_ce matching", "train done", "score ", "all stages", "PASS", "FAIL")
start = log.rfind("=== Fold B")
print("\n".join(l for l in log.splitlines() if l.lstrip("[0123456789:] ").startswith(keys) or "pairs/s" in l[-40:])[-6000:])
print(log[start:start + 4000] if start >= 0 else "no fold-B table yet")
print(subprocess.run(f"ls -la {OUT_CE}; du -sh {CE_DIR}", shell=True, capture_output=True, text=True).stdout)